# Add Understat xG Data

Adds Understat advanced metrics to your existing `player_db.csv`.

**Run this AFTER `build_player_database.ipynb` completes.**

In [77]:
import pandas as pd
import numpy as np
import os
import unicodedata
import warnings
warnings.filterwarnings('ignore')

In [78]:
RAW_DIR = os.path.join('..', 'data', 'raw')
DB_PATH = os.path.join('..', 'data', 'processed', 'player_db.csv')
UNDERSTAT_FOLDER = os.path.join(RAW_DIR, 'UnderstatData')

LEAGUE_MAP = {
    'de Bundesliga': 'Bundesliga',
    'fr Ligue 1': 'Ligue 1',
    'es La Liga': 'La Liga',
    'eng Premier League': 'Premier League',
    'it Serie A': 'Serie A',
}

In [79]:
def normalise_name(name):
    if pd.isna(name):
        return ''
    name = str(name).strip().lower()
    name = unicodedata.normalize('NFD', name)
    return ''.join(c for c in name if unicodedata.category(c) != 'Mn')

## 1. Load Existing Database

In [80]:
if not os.path.exists(DB_PATH):
    print(f"Database not found: {DB_PATH}")
    print("Run build_player_database.ipynb first")
else:
    master = pd.read_csv(DB_PATH)
    print(f"✓ Loaded database: {len(master):,} rows")
    master['name_clean'] = master['name'].apply(normalise_name)
    master['league_understat'] = master['league_comp'].map(LEAGUE_MAP)
    display(master.head())

✓ Loaded database: 13,640 rows


,rank,name,Nation,position,club,league_comp,Age,Born,appearances,Starts,...,CrdR,Gls_90,Ast_90,G+A_90,G-PK_90,G+A-PK_90,season,rating,name_clean,league_understat
0,2700,Aaron Ciammaglichella,it ITA,MF,Torino,it Serie A,19.0,2005.0,1,0,...,0,0.00,0.00,0.00,0.00,0.00,2024-2025,NaN,aaron ciammaglichella,Serie A
1,1713,Aaron Connolly,ie IRL,FW/MF,Brighton,eng Premier League,20.0,2000.0,17,9,...,0,0.23,0.11,0.34,0.23,0.34,2020-2021,NaN,aaron connolly,Premier League
2,2318,Aaron Connolly,ie IRL,FW,Brighton,eng Premier League,21.0,2000.0,4,1,...,0,0.00,0.00,0.00,0.00,0.00,2021-2022,NaN,aaron connolly,Premier League
3,67,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,30.0,1989.0,36,36,...,0,0.00,0.23,0.23,0.00,0.23,2020-2021,6.97,aaron cresswell,Premier League
4,269,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,31.0,1989.0,31,31,...,0,0.07,0.10,0.17,0.07,0.17,2021-2022,7.10,aaron cresswell,Premier League


## Manual Name Mappings

Handle specific nickname cases and alternate spellings.

In [81]:
MANUAL_NAME_MAPPINGS = {
    "obite n'dicka": "eric ndicka",
    "pascu": "jorge pascual",
    "przemysław płacheta": "przemyslaw placheta",
    "simeon nwankwo": "simy",
    "tachi": "alberto rodríguez",
    "vassilis lambropoulos": "vasilios lampropoulos",
    "yassine bounou": "bono",
    "cifu": "cifuentes",
    "cucho": "juan camilo hernández",
    "emmanuel amankwaa akurugu": "koffi",
    "lauti": "lautaro de leon",
    "mathias jørgensen": "zanka",
    "carmona": "josé ángel carmona"
}

CONTEXT_MAPPINGS = [
    ('rodri', '2020-2021', 'betis', 'rodrigo sánchez'),
    ('rodri', '2021-2022', 'betis', 'rodrigo sánchez'),
    ('rodri', '2022-2023', 'betis', 'rodrigo sánchez'),
    ('rodri', '2023-2024', 'betis', 'rodrigo sánchez'),
    ('rodri', '2024-2025', 'betis', 'rodrigo sánchez'),
    ('ridle baku', '2024-2025', 'leipzig', 'bote baku'),
    ('terem moffi', '2022-2023', 'lorient', 'terem igobor moffi')
]

if 'master' in locals():
    print("Applying manual name mappings...\n")
    
    for old_name, new_name in MANUAL_NAME_MAPPINGS.items():
        mask = master['name_clean'] == old_name
        count = mask.sum()
        if count > 0:
            master.loc[mask, 'name_clean'] = new_name
            print(f"  ✓ Mapped '{old_name}' → '{new_name}' ({count} rows)")
    
    for name, season, club_keyword, mapped_name in CONTEXT_MAPPINGS:
        mask = (
            (master['name_clean'] == name) &
            (master['season'] == season) &
            (master['club'].str.lower().str.contains(club_keyword, na=False))
        )
        count = mask.sum()
        if count > 0:
            master.loc[mask, 'name_clean'] = mapped_name
            print(f"  ✓ Mapped '{name}' → '{mapped_name}' for {season} {club_keyword} ({count} rows)")
    
    print("\nManual mappings complete.")
else:
    print("master not loaded yet")

Applying manual name mappings...

  ✓ Mapped 'obite n'dicka' → 'eric ndicka' (5 rows)
  ✓ Mapped 'pascu' → 'jorge pascual' (2 rows)
  ✓ Mapped 'przemysław płacheta' → 'przemyslaw placheta' (1 rows)
  ✓ Mapped 'simeon nwankwo' → 'simy' (3 rows)
  ✓ Mapped 'tachi' → 'alberto rodríguez' (2 rows)
  ✓ Mapped 'vassilis lambropoulos' → 'vasilios lampropoulos' (2 rows)
  ✓ Mapped 'yassine bounou' → 'bono' (4 rows)
  ✓ Mapped 'cifu' → 'cifuentes' (1 rows)
  ✓ Mapped 'cucho' → 'juan camilo hernández' (3 rows)
  ✓ Mapped 'emmanuel amankwaa akurugu' → 'koffi' (2 rows)
  ✓ Mapped 'lauti' → 'lautaro de leon' (1 rows)
  ✓ Mapped 'mathias jørgensen' → 'zanka' (3 rows)
  ✓ Mapped 'carmona' → 'josé ángel carmona' (4 rows)


  ✓ Mapped 'rodri' → 'rodrigo sánchez' for 2020-2021 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodrigo sánchez' for 2021-2022 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodrigo sánchez' for 2022-2023 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodrigo sánchez' for 2023-2024 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodrigo sánchez' for 2024-2025 betis (1 rows)
  ✓ Mapped 'ridle baku' → 'bote baku' for 2024-2025 leipzig (1 rows)
  ✓ Mapped 'terem moffi' → 'terem igobor moffi' for 2022-2023 lorient (1 rows)

Manual mappings complete.


## 2. Load Understat Data

In [82]:
def load_understat_data():
    """
    Load Understat xG CSVs.
    Handles players with multiple teams (transfers) - takes the most recent team.
    """
    year_to_season = {
        '2020': '2020-2021',
        '2021': '2021-2022',
        '2022': '2022-2023',
        '2023': '2023-2024',
        '2024': '2024-2025',
    }
    
    league_folders = {
        'Bundesliga': 'Bundesliga',
        'La_Liga': 'La Liga',
        'Ligue_1': 'Ligue 1',
        'Premier_League': 'Premier League',
        'Serie_A': 'Serie A',
    }
    
    if not os.path.exists(UNDERSTAT_FOLDER):
        print(f"Understat folder not found: {UNDERSTAT_FOLDER}")
        return pd.DataFrame()
    
    all_xg = []
    
    for folder_name, league_name in league_folders.items():
        league_path = os.path.join(UNDERSTAT_FOLDER, folder_name)
        if not os.path.exists(league_path):
            print(f"Folder not found: {folder_name}/")
            continue
        
        print(f"  Loading {league_name}...")
        count = 0
        
        for year, season in year_to_season.items():
            filepath = os.path.join(league_path, f"{year}.csv")
            if not os.path.exists(filepath):
                continue
            
            try:
                rows = []
                with open(filepath, 'r', encoding='utf-8') as f:
                    _ = f.readline()
                    
                    for line in f:
                        line = line.replace("&#039;", "'")
                        line = line.strip()
                        
                        if line.startswith('"') and line.endswith('"'):
                            line = line[1:-1]
                        
                        line = line.rstrip(',')
                        
                        parts = line.split(';')
                        
                        parts = [p.strip('"').strip() for p in parts]
                        
                        if len(parts) < 10:
                            continue
                        
                        if len(parts) > 15:
                            parts[2] = parts[3] if len(parts) > 3 else parts[2]
                            parts = parts[:3] + parts[4:]
                        
                        if len(parts) == 15:
                            rows.append(parts)
                
                if not rows:
                    print(f"No valid data in {year}.csv")
                    continue
                
                columns = ['number', 'player', 'team', 'apps', 'min', 'goals', 'a',
                          'xG', 'xA', 'xG90', 'xA90', 'xG90xA90', 'NPxG90xA90',
                          'xGChain90', 'xGBuildup90']
                
                df = pd.DataFrame(rows, columns=columns)
                
                numeric_cols = ['apps', 'min', 'goals', 'a', 'xG', 'xA', 'xG90',
                               'xA90', 'xG90xA90', 'NPxG90xA90', 'xGChain90', 'xGBuildup90']
                for col in numeric_cols:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                
                df['season'] = season
                df['league'] = league_name
                all_xg.append(df)
                count += len(df)
                
            except Exception as e:
                print(f"Error reading {year}.csv: {e}")
                continue
        
        if count:
            print(f"    → {count:,} rows")
    
    if not all_xg:
        print("No Understat data found")
        return pd.DataFrame()
    
    combined = pd.concat(all_xg, ignore_index=True)
    combined.rename(columns={'player': 'name'}, inplace=True)
    combined['name_clean'] = combined['name'].apply(normalise_name)
    
    print(f"\n✓ Total Understat data: {len(combined):,} rows")
    return combined

print("Loading Understat data...\n")
understat = load_understat_data()

if not understat.empty:
    display(understat.head())

Loading Understat data...

  Loading Bundesliga...
    → 2,488 rows
  Loading La Liga...
    → 2,944 rows
  Loading Ligue 1...
    → 2,816 rows
  Loading Premier League...
    → 2,747 rows
  Loading Serie A...
    → 2,968 rows

✓ Total Understat data: 13,963 rows


,number,name,team,apps,min,goals,a,xG,xA,xG90,xA90,xG90xA90,NPxG90xA90,xGChain90,xGBuildup90,season,league,name_clean
0,1,Robert Lewandowski,Bayern Munich,29,2467,41,7,32.08,4.82,1.17,0.18,1.35,1.10,1.16,0.21,2020-2021,Bundesliga,robert lewandowski
1,2,André Silva,Eintracht Frankfurt,32,2787,28,5,25.60,5.47,0.83,0.18,1.00,0.83,0.86,0.13,2020-2021,Bundesliga,andre silva
2,3,Erling Haaland,Borussia Dortmund,28,2416,27,6,23.60,4.04,0.88,0.15,1.03,0.92,1.01,0.22,2020-2021,Bundesliga,erling haaland
3,4,Andrej Kramaric,Hoffenheim,28,2386,20,5,15.53,4.07,0.59,0.15,0.74,0.60,0.68,0.20,2020-2021,Bundesliga,andrej kramaric
4,5,Wout Weghorst,Wolfsburg,34,2954,20,8,18.31,5.43,0.56,0.17,0.72,0.65,0.74,0.18,2020-2021,Bundesliga,wout weghorst


## 3. Merge xG Data

In [83]:
if not understat.empty and 'master' in locals():
    print("Merging xG data on [name, season, league]...\n")
    
    understat['team_clean'] = understat['team'].str.lower().str.strip()
    master['team_clean'] = master['club'].str.lower().str.strip()
    
    def split_name_words(name):
        """Split name on spaces and apostrophes for better matching."""
        import re
        if pd.isna(name):
            return set()
        words = re.split(r"[\s'\-]+", str(name).lower())
        return set(w for w in words if len(w) > 1)
    
    xg_cols = ['name_clean', 'season', 'league', 'team', 'goals', 'a',
               'xG', 'xA', 'xG90', 'xA90', 'xG90xA90', 'NPxG90xA90', 
               'xGChain90', 'xGBuildup90', 'team_clean']
    
    before_rows = len(master)
    
    print("Stage 1: Exact name matching...")
    master = master.merge(
        understat[xg_cols],
        left_on=['name_clean', 'season', 'league_understat'],
        right_on=['name_clean', 'season', 'league'],
        how='left',
        suffixes=('', '_understat')
    )
    master = master.drop(columns=['league'], errors='ignore')
    
    if "team_clean_understat" in master.columns:
        master["team_match_score"] = master.apply(
            lambda row: len(set(str(row["team_clean"]).split()) & 
                           set(str(row.get("team_clean_understat", "")).split()))
            if pd.notna(row.get("xG")) else -1,
            axis=1
        )
        master = master.sort_values("team_match_score", ascending=False)
        master = master.drop_duplicates(subset=["name_clean", "season", "club"], keep="first")
        master = master.drop(columns=["team_match_score"], errors="ignore")

    exact_matches = master['xG'].notna().sum()
    print(f"  ✓ Exact matches: {exact_matches:,}")
    
    unmatched = master[master['xG'].isna()].copy()
    
    if len(unmatched) > 0:
        print(f"\nStage 2: Fuzzy matching for {len(unmatched):,} unmatched players...")
        print("  Matching on: partial name + team + goals + assists")
        
        fuzzy_matches = []
        
        for _, player_row in unmatched.iterrows():
            player_words = split_name_words(player_row['name_clean'])
            player_season = player_row['season']
            player_league = player_row['league_understat']
            player_goals = player_row['goals']
            player_assists = player_row['assists']
            player_team = player_row['team_clean']
            
            candidates = understat[
                (understat['season'] == player_season) & 
                (understat['league'] == player_league)
            ].copy()
            
            best_match = None
            best_score = 0
            
            for _, und_row in candidates.iterrows():
                und_words = split_name_words(und_row['name_clean'])
                score = 0
                
                name_overlap = len(player_words & und_words)
                if name_overlap == 0:
                    continue
                score += name_overlap * 10
                
                if pd.notna(player_goals) and pd.notna(und_row['goals']):
                    if abs(float(player_goals) - float(und_row['goals'])) < 0.5:
                        score += 100
                
                if pd.notna(player_assists) and pd.notna(und_row['a']):
                    if abs(float(player_assists) - float(und_row['a'])) < 0.5:
                        score += 50
                
                und_team_words = set(und_row['team_clean'].split())
                player_team_words = set(player_team.split())
                team_overlap = len(und_team_words & player_team_words)
                if team_overlap > 0:
                    score += team_overlap * 20

                if score > best_score and score >= 50:
                    best_score = score
                    best_match = und_row
            
            if best_match is not None:
                fuzzy_matches.append({
                    'master_idx': player_row.name,
                    'fbref_name': player_row['name'],
                    'fbref_team': player_row['club'],
                    'fbref_goals': player_goals,
                    'fbref_assists': player_assists,
                    'understat_name': best_match['name'],
                    'understat_team': best_match['team'],
                    'understat_goals': best_match['goals'],
                    'understat_assists': best_match['a'],
                    'score': best_score,
                    'xG': best_match['xG'],
                    'xA': best_match['xA'],
                    'xG90': best_match['xG90'],
                    'xA90': best_match['xA90'],
                    'xG90xA90': best_match['xG90xA90'],
                    'NPxG90xA90': best_match['NPxG90xA90'],
                    'xGChain90': best_match['xGChain90'],
                    'xGBuildup90': best_match['xGBuildup90'],
                })
        
        if fuzzy_matches:
            print(f"  ✓ Fuzzy matched: {len(fuzzy_matches)} additional players")
            print(f"\n  Examples (showing match confidence):")
            for match in sorted(fuzzy_matches, key=lambda x: x['score'], reverse=True)[:10]:
                print(f"    [{match['score']:3.0f}] '{match['fbref_name']}' ({match['fbref_team']}) ← '{match['understat_name']}' ({match['understat_team']})")
                print(f"         Goals: {match['fbref_goals']} vs {match['understat_goals']} | Assists: {match['fbref_assists']} vs {match['understat_assists']}")
            
            for match in fuzzy_matches:
                idx = match['master_idx']
                for col in ['xG', 'xA', 'xG90', 'xA90', 'xG90xA90', 'NPxG90xA90', 'xGChain90', 'xGBuildup90']:
                    master.loc[idx, col] = match[col]
    
    master = master.drop(columns=['team_clean', 'team_clean_understat', 'team_understat', 
                                   'goals_understat', 'a'], errors='ignore')
    
    after_rows = len(master)
    
    if after_rows != before_rows:
        print(f"Row count changed: {before_rows:,} → {after_rows:,}")
    
    matched = master['xG'].notna().sum()
    print(f"\n{'='*60}")
    print(f"✓ Total xG data merged: {matched:,} / {len(master):,} ({matched/len(master)*100:.1f}%)")
    print(f"{'='*60}")
    
    valid = master[master['xG'].notna() & master['goals'].notna()]
    if len(valid) > 100:
        corr = valid[['goals', 'xG']].corr().iloc[0, 1]
        print(f"  Correlation goals vs xG: {corr:.3f} (should be 0.7-0.9)")
    
    print("\nxG coverage by season:")
    display(master.groupby('season')['xG'].agg(['count', lambda x: x.notna().sum()]))
else:
    print("Cannot merge - missing data")

Merging xG data on [name, season, league]...

Stage 1: Exact name matching...
  ✓ Exact matches: 12,280

Stage 2: Fuzzy matching for 1,356 unmatched players...
  Matching on: partial name + team + goals + assists
  ✓ Fuzzy matched: 1354 additional players

  Examples (showing match confidence):
    [220] 'Junior Dina Ebimbe' (Eintracht Frankfurt) ← 'Eric Junior Dina Ebimbe' (Eintracht Frankfurt)
         Goals: 3 vs 3 | Assists: 1 vs 1
    [220] 'Mohamed Ali Cho' (Real Sociedad) ← 'Mohamed Ali-Cho' (Real Sociedad)
         Goals: 1 vs 1 | Assists: 2 vs 2
    [220] 'Junior Dina Ebimbe' (Eintracht Frankfurt) ← 'Eric Junior Dina Ebimbe' (Eintracht Frankfurt)
         Goals: 5 vs 5 | Assists: 3 vs 3
    [220] 'Junior Dina Ebimbe' (Eintracht Frankfurt) ← 'Eric Junior Dina Ebimbe' (Eintracht Frankfurt)
         Goals: 0 vs 0 | Assists: 1 vs 1
    [220] 'Ahmed Hegazi' (West Bromwich Albion) ← 'Ahmed Hegazy' (West Bromwich Albion)
         Goals: 0 vs 0 | Assists: 0 vs 0
    [220] 'Jesurun Rak

,count,<lambda_0>
season,,
2020-2021,2703,2703
2021-2022,2791,2791
2022-2023,2724,2724
2023-2024,2709,2709
2024-2025,2707,2707


## 4. Verify Merge

In [84]:
if 'master' in locals() and 'xG' in master.columns:
    print("Sample players with xG data:")
    sample = master[master['xG'].notna()].head(10)
    display(sample[['name', 'club', 'season', 'goals', 'xG', 'xG90', 'xGChain90']])

Sample players with xG data:


,name,club,season,goals,xG,xG90,xGChain90
8628,Matheus Pereira,West Bromwich Albion,2020-2021,11,6.95,0.24,0.38
3974,Filip Krovinović,West Bromwich Albion,2020-2021,0,0.22,0.04,0.12
7350,Lee Peltier,West Bromwich Albion,2020-2021,0,0.00,0.00,0.07
10888,Rekeem Harper,West Bromwich Albion,2020-2021,0,0.03,0.10,0.13
11230,Romaine Sawyers,West Bromwich Albion,2020-2021,0,0.15,0.01,0.11
7206,Kyle Edwards,West Bromwich Albion,2020-2021,0,0.12,0.10,0.16
7205,Kyle Bartley,West Bromwich Albion,2020-2021,3,2.60,0.09,0.09
4722,Hal Robson-Kanu,West Bromwich Albion,2020-2021,2,1.50,0.26,0.39
1803,Branislav Ivanović,West Bromwich Albion,2020-2021,0,0.46,0.05,0.04
9997,Okay Yokuşlu,West Bromwich Albion,2020-2021,0,0.72,0.05,0.19


## 5. Save Updated Database

In [85]:
if 'master' in locals():
    master = master.drop(columns=['name_clean', 'league_understat'], errors='ignore')
    master = master.sort_values(['name', 'season']).reset_index(drop=True)
    
    master.to_csv(DB_PATH, index=False)
    
    print("=" * 60)
    print(f"✓ SAVED: {DB_PATH}")
    print("=" * 60)
    print(f"Rows: {len(master):,}")
    print(f"Columns: {len(master.columns)}")
    
    xg_cols_added = [c for c in master.columns if c.startswith('xG') or c.startswith('xA') or 'NPxG' in c]
    print(f"\nxG columns added: {xg_cols_added}")
else:
    print("Nothing to save")

✓ SAVED: ..\data\processed\player_db.csv
Rows: 13,636
Columns: 36

xG columns added: ['xG', 'xA', 'xG90', 'xA90', 'xG90xA90', 'NPxG90xA90', 'xGChain90', 'xGBuildup90']


## 6. Final Preview

In [86]:
if 'master' in locals():
    print("Updated database:")
    master.info()
    print("\nSample:")
    display(master.head())

Updated database:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13636 entries, 0 to 13635
Data columns (total 36 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   rank         13636 non-null  int64  
 1   name         13636 non-null  object 
 2   Nation       13631 non-null  object 
 3   position     13636 non-null  object 
 4   club         13636 non-null  object 
 5   league_comp  13636 non-null  object 
 6   Age          13634 non-null  float64
 7   Born         13634 non-null  float64
 8   appearances  13636 non-null  int64  
 9   Starts       13636 non-null  int64  
 10  minutes      13636 non-null  int64  
 11  90s          13636 non-null  float64
 12  goals        13636 non-null  int64  
 13  assists      13636 non-null  int64  
 14  G+A          13636 non-null  int64  
 15  G-PK         13636 non-null  int64  
 16  PK           13636 non-null  int64  
 17  PKatt        13636 non-null  int64  
 18  CrdY         13636 non-null 

,rank,name,Nation,position,club,league_comp,Age,Born,appearances,Starts,...,rating,team,xG,xA,xG90,xA90,xG90xA90,NPxG90xA90,xGChain90,xGBuildup90
0,2700,Aaron Ciammaglichella,it ITA,MF,Torino,it Serie A,19.0,2005.0,1,0,...,NaN,Torino,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,1713,Aaron Connolly,ie IRL,FW/MF,Brighton,eng Premier League,20.0,2000.0,17,9,...,NaN,Brighton,4.46,0.16,0.50,0.02,0.52,0.52,0.54,0.02
2,2318,Aaron Connolly,ie IRL,FW,Brighton,eng Premier League,21.0,2000.0,4,1,...,NaN,Brighton,0.60,0.38,0.35,0.23,0.58,0.58,0.60,0.02
3,67,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,30.0,1989.0,36,36,...,6.97,West Ham,0.88,7.39,0.03,0.21,0.23,0.23,0.30,0.24
4,269,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,31.0,1989.0,31,31,...,7.10,West Ham,0.91,3.67,0.03,0.12,0.15,0.15,0.35,0.32
